# Dataset Statistics: QC Metrics

Evaluates three complementary quality dimensions that characterise the reliability and spatial consistency of measurements across all participating laboratories:

1. **Missingness / completeness** — how completely each feature is detected across samples, and how completely each sample covers the feature space, at both single-cell and QC injection level.
2. **Plate heatmap** — spatial distribution of identification counts across each physical 16 × 24 well plate, overlaying single-cell and QC injections to expose within-plate gradients or positional effects.
3. **Retention time (RT) stability** — the shape of the elution profile (proportion of identifications per RT bin) across samples, used to detect gradient failures or systematic shifts between labs.

These metrics are used in **Figure 4** of the manuscript. The completeness values computed here are also stored back into the AnnData objects and used downstream as a sample-level QC filter.

**Inputs**

| File | Description |
|---|---|
| `Data/AnnData/adata_precursor.h5ad` | Precursor-level quantification, single cells |
| `Data/AnnData/adata_protein.h5ad` | Protein-level quantification, single cells |
| `Data/AnnData/adata_precursor_qc.h5ad` | Precursor-level quantification, QC injections |
| `Data/AnnData/adata_protein_qc.h5ad` | Protein-level quantification, QC injections |
| `Data/Parquet/search_results.parquet` | Long-format search results (needed for per-precursor RT values) |

**Outputs** (`Figures/Figure_4/`)

| Panel | File | Description |
|---|---|---|
| A | `RT_Stability/RT_stability_overview_SC.png` | RT stability overview, single-cell conditions, coloured by lab |
| B | `RT_Stability/RT_stability_overview_QC.png` | RT stability overview, QC conditions, coloured by lab |
| C | `Plate_Heatmap/plate_heatmap_protein_{lab}.png` | Per-lab protein ID plate heatmap |
| D | `Plate_Heatmap/plate_heatmap_precursor_{lab}.png` | Per-lab precursor ID plate heatmap |
| E | `Completeness/completeness_precursors_SC.png` | Precursor completeness curves, single cells |
| E | `Completeness/completeness_precursors_QC.png` | Precursor completeness curves, QC injections |
| F | `Completeness/completeness_proteins_SC.png` | Protein completeness curves, single cells |
| F | `Completeness/completeness_proteins_QC.png` | Protein completeness curves, QC injections |
| Supp. | `RT_Stability/RT_stability_{lab}.png` | Per-lab RT stability coloured by condition |

**AnnData updates** — the following columns are written back before saving:

| Object | Slot | Column | Description |
|---|---|---|---|
| precursor / protein | `obs` | `completeness_sample_SC` | Fraction of lab-detected features found in each single cell |
| precursor / protein | `obs` | `completeness_sample_QC` | Fraction of lab-detected features found in each QC injection |
| precursor / protein | `var` | `completeness_feature_{lab}_SC/QC` | Per-lab detection rate across samples for each feature |
| precursor / protein | `obs` | `completeness_outlier` | `True` if sample fails the specified completeness thresholds |

---

## Imports

In [ ]:
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
import anndata

from SCI_py.metrics.completeness import compute_completeness, compute_combined_curve, store_completeness
from SCI_py.plot.completeness import plot_completeness
from SCI_py.plot.plate_heatmap import plot_plate_heatmap
from SCI_py.metrics.rt_stability import compute_rt_proportions
from SCI_py.plot.rt_stability import plot_rt_stability
from SCI_py.constants.maps import LABS
from SCI_py.data.subset import get_proteins_by_condition

tqdm.pandas()

## 0. Path Configuration

Input AnnData is read from `ROOT_OUT/AnnData/base`. Figures are written to `FIGURES_OUT/Figure_4/` with one sub-directory per panel (`RT_Stability/`, `Plate_Heatmap/`, `Completeness/`). Both root directories can be overridden via environment variables (`SCI_DATA_DIR`, `SCI_FIGURES_DIR`) for running the notebook on different machines or when the user wants to specify their own output directories

In [2]:
NOTEBOOK_DIR = Path().resolve()
ROOT_OUT      = NOTEBOOK_DIR / "../../Data"
ROOT_OUT      = ROOT_OUT.resolve()
ROOT_OUT.mkdir(parents=True, exist_ok=True)

# Allow user override via environment variable
ROOT_OUT = Path(os.environ.get("SCI_DATA_DIR", ROOT_OUT))

ROOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {ROOT_OUT}")

FIGURES_OUT = NOTEBOOK_DIR / "../../Figures"
FIGURES_OUT = FIGURES_OUT.resolve()
FIGURES_OUT.mkdir(parents=True, exist_ok=True)

# Overwrite with environment variable
FIGURES_OUT = Path(os.environ.get("SCI_FIGURES_DIR", FIGURES_OUT))
print(f"Figures directory: {FIGURES_OUT}")

ANNDATA_IN = os.path.join(
    ROOT_OUT,
    'AnnData',
    'base'
)

Output directory: /home/sam/SCI/SCI/Data
Figures directory: /home/sam/SCI/SCI/Figures


In [3]:
SHOW_FIGURES = False

## 1. Load Data

Four AnnData objects are loaded: single-cell data at precursor level (3,114 samples × 72,061 precursors) and protein level (3,114 × 7,231), plus the equivalent QC injection objects. 

Identification counts (`n_precursors`, `n_proteins`) are computed for the QC objects here so they are available for the plate heatmap visualisation.

In [5]:
adata_precursor = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor.h5ad')
)
adata_protein = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_protein.h5ad')
)

adata_precursor_qc = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_precursor_qc.h5ad')
)
adata_protein_qc = anndata.io.read_h5ad(
    os.path.join(ANNDATA_IN, 'adata_protein_qc.h5ad')
)

In [6]:
# Add counts
adata_precursor_qc.obs['n_precursors'] = np.sum(~np.isnan(
    adata_precursor_qc[:, ~adata_precursor_qc.var['Contaminant']].X
), axis=1)
adata_protein_qc.obs['n_proteins'] = np.sum(~np.isnan(
    adata_protein_qc[:, ~adata_protein_qc.var['Contaminant']].X
), axis=1)

In [6]:
adata_precursor

AnnData object with n_obs × n_vars = 3114 × 72061
    obs: 'sample_name', 'plate_id', 'plate_idx', 'source name', 'comment[X position]', 'comment[Y position]', 'comment[Plate Number]', 'comment[Gradient time]', 'comment[thawing timepoint]', 'comment[plate arrival]', 'injection_order', 'institute_name', 'Reinjection', 'Diameter', 'Elongation', 'Circularity', 'Doublet', 'Dividing', 'Dying', 'CellInBlank', 'comment[plate arrival]_date', 'comment[plate arrival]_time', 'comment[plate arrival]_datetime', 'comment[Gradient time]_date', 'comment[Gradient time]_time', 'comment[Gradient time]_datetime', 'instrument_type'
    var: 'PG.ProteinGroups', 'PG.ProteinAccessions', 'PEP.StrippedSequence', 'PEP.IsProteotypic', 'EG.ModifiedSequence', 'FG.Charge', 'FG.LabeledSequence', 'PG.Pvalue', 'PG.Qvalue', 'FG.Qvalue', 'EG.GlobalPrecursorQvalue', 'PG.Cscore', 'EG.ApexRT', 'EG.RTPredicted', 'EG.iRTEmpirical', 'EG.iRTPredicted', 'FG.PrecWindowNumber', 'FG.CScore', 'Contaminant'
    layers: 'EG.ApexRT', '

In [7]:
adata_protein

AnnData object with n_obs × n_vars = 3114 × 7231
    obs: 'sample_name', 'plate_id', 'plate_idx', 'source name', 'comment[X position]', 'comment[Y position]', 'comment[Plate Number]', 'comment[Gradient time]', 'comment[thawing timepoint]', 'comment[plate arrival]', 'injection_order', 'institute_name', 'Reinjection', 'Diameter', 'Elongation', 'Circularity', 'Doublet', 'Dividing', 'Dying', 'CellInBlank', 'comment[plate arrival]_date', 'comment[plate arrival]_time', 'comment[plate arrival]_datetime', 'comment[Gradient time]_date', 'comment[Gradient time]_time', 'comment[Gradient time]_datetime', 'instrument_type'
    var: 'PG.ProteinAccessions', 'PEP.StrippedSequence', 'PEP.IsProteotypic', 'EG.ModifiedSequence', 'FG.Charge', 'FG.LabeledSequence', 'FG.PrecursorIon', 'PG.Pvalue', 'PG.Qvalue', 'FG.Qvalue', 'EG.GlobalPrecursorQvalue', 'PG.Cscore', 'EG.ApexRT', 'EG.RTPredicted', 'EG.iRTEmpirical', 'EG.iRTPredicted', 'FG.PrecWindowNumber', 'FG.CScore', 'Contaminant'
    layers: 'PG.Cscore', 'PG

## 2. QC Tiers

Quality assessment is structured in three tiers that together span the spatial, quantitative, and chromatographic dimensions of data quality. Each tier is computed separately for **single-cell conditions** (K562, HEK293T) and **QC injection conditions** (QC - HeLa, site-specific QC), allowing cross-context comparison of the same metric.

### 2.1. Missingness Pattern — Completeness Curves

Completeness curves show the trade-off between stringency and data retention when applying a missing-value filter. Two complementary perspectives are computed per plate:

- **Feature-level completeness** (left panel) — for each threshold *t* (0–100%), the fraction of features detected in ≥ *t*% of samples. A feature with 50% completeness is quantified in half the cells. The curve's shape reveals whether missingness is diffuse (gradual decline) or concentrated in a few stochastically undetected features (steep drop near 0%).
- **Sample-level completeness** (right panel) — for each threshold *t*, the fraction of samples that have ≥ *t*% of the plate's detectable features. A sample at 25% completeness has quantified one quarter of all uniquely detected features. This identifies near-empty injections (failed electrospray, clogged column, dispensing error...).

Each plate is shown as a coloured line; the dashed black line represents the combined curve across all plates. The 50% reference line provides a natural anchor for comparison.

All four objects (precursor + protein × SC + QC conditions) are processed, and the resulting per-sample and per-feature completeness values are stored back into `adata.obs` and `adata.var` via `store_completeness`, with suffixes `_SC` and `_QC` to keep the two condition sets distinguishable.

#### Single-cell conditions (K562, HEK293T)

Completeness is evaluated only within the two primary cell-line conditions. Per-sample completeness is stored as `completeness_sample_SC` in `adata.obs`; per-feature detection rates per plate are stored as `completeness_feature_{plate}_SC` in `adata.var`.

In [9]:
# Precursor
conditions = ['K562', 'HEK293T']
thresholds, retention, samples = compute_completeness(adata_precursor, conditions)
combined_feat = compute_combined_curve(adata_precursor, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_precursor, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds=thresholds,
    retention_curves=retention,
    sample_curves=samples,
    combined_feature=combined_feat,
    combined_sample=combined_samp,
    label='Precursors',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_precursors_SC.png',
    show=SHOW_FIGURES
)
store_completeness(
    adata=adata_precursor,
    conditions=conditions,
    conditions_suffix='SC'
)

# Protein
thresholds, retention, samples = compute_completeness(adata_protein, conditions)
combined_feat = compute_combined_curve(adata_protein, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_protein, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds=thresholds,
    retention_curves=retention,
    sample_curves=samples,
    combined_feature=combined_feat,
    combined_sample=combined_samp,
    label='Proteins',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_proteins_SC.png',
    show=SHOW_FIGURES
)
store_completeness(
    adata=adata_protein,
    conditions=conditions,
    conditions_suffix='SC'
)

#### QC injection conditions (QC - HeLa, site-specific QC)

The same completeness analysis is repeated for the HeLa and site-specific QC injections. Because QC injections use bulk material (not single cells), completeness should be substantially higher — any QC injection with low completeness is a stronger signal of an instrument or LC problem than the same completeness in a single cell. Results stored with suffix `_QC`.

In [10]:
# Precursor
conditions = ['QC - Hela', 'Site specific QC']
thresholds, retention, samples = compute_completeness(adata_precursor, conditions)
combined_feat = compute_combined_curve(adata_precursor, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_precursor, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds=thresholds,
    retention_curves=retention,
    sample_curves=samples,
    combined_feature=combined_feat,
    combined_sample=combined_samp,
    label='Precursors',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_precursors_QC.png',
    show=SHOW_FIGURES
)
store_completeness(
    adata=adata_precursor,
    conditions=conditions,
    conditions_suffix='QC'
)

# Protein
thresholds, retention, samples = compute_completeness(adata_protein, conditions)
combined_feat = compute_combined_curve(adata_protein, conditions, axis=0, thresholds=thresholds)
combined_samp = compute_combined_curve(adata_protein, conditions, axis=1, thresholds=thresholds)

plot_completeness(
    thresholds=thresholds,
    retention_curves=retention,
    sample_curves=samples,
    combined_feature=combined_feat,
    combined_sample=combined_samp,
    label='Proteins',
    conditions=conditions,
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'Completeness'),
    filename='completeness_proteins_QC.png',
    show=SHOW_FIGURES
)
store_completeness(
    adata=adata_protein,
    conditions=conditions,
    conditions_suffix='QC'
)

### 2.2. Plate Heatmap

Visualises per-well identification counts spatially on the physical plate layout. The plate is drawn as a 16 (rows) × 24 (columns) grid; each well is rendered as a coloured rectangle with the identification count printed inside.

**Colour encoding** — wells are filled with the condition colour from `CONDITION_COLORS`. The colour alpha (opacity) is linearly scaled from 0.3 (minimum count) to 1.0 (maximum count within the plate), so the intensity of the colour directly encodes how many features were identified. Empty or missing wells are shown at α = 0.1.

**Section boundaries** — white grid lines separate the distinct well sections of the SCI plate design (e.g. cell wells, blank wells, QC wells), making it easy to verify that identification depth tracks expected plate layout patterns.

**QC overlay** — the `extra_adata` argument injects the quantitative QC injection observations into the same plate visualisation. Observations from these wells, which were run separately, are merged with the single-cell plate on (X, Y) position, so the full physical plate is shown in a single figure. This is particularly useful for diagnosing whether low-count regions are confined to cell wells (dispensing issue) or span QC wells too (some other issue).

One figure is produced per plate, at both protein and precursor level.

In [14]:
plot_plate_heatmap(
    adata_protein,
    count_col   = 'n_proteins',
    extra_adata = adata_protein_qc,
    label       = 'Proteins',
    save_dir    = os.path.join(FIGURES_OUT, 'Figure_4', 'Plate_Heatmap'),
    filename_fmt = 'plate_heatmap_protein_{lab}.png',
    show=SHOW_FIGURES
)

plot_plate_heatmap(
    adata_precursor,
    count_col   = 'n_precursors',
    extra_adata = adata_precursor_qc,
    label       = 'Precursors',
    save_dir    = os.path.join(FIGURES_OUT, 'Figure_4', 'Plate_Heatmap'),
    filename_fmt = 'plate_heatmap_precursor_{lab}.png',
    show=SHOW_FIGURES
)

### 2.3. Retention Time Stability

Chromatographic stability is assessed by computing the proportion of identified precursors falling into each RT bin for every sample, then comparing the distribution shapes across samples and plates. A stable chromatographic run produces an elution profile of consistent shape; deviations indicate gradient drift, injection failures, or carry-over. Here, we do include the contaminant proteins as we are solely interested in the identifications across the elution profile, irrespective of the source of these proteins.

**Method** — for each sample, the `EG.ApexRT` values, stored in a layer of the AnnData objects, are normalised to [0, 1] within that sample (`normalize_rt=True`), binned into 25 equal-width bins, and the fraction of identifications per bin is computed. Normalising per sample removes the effect of absolute gradient length differences between plates, making elution profiles directly comparable. In this study, normalising is irrelevant since all samples were run with the same gradient lenghts.

**Outlier exclusion** — samples flagged as `completeness_outlier` (< 25% completeness in the SC and QC conditions; see threshold cell below) are excluded from RT analysis. Near-empty injections produce noisy, uninformative RT profiles and would otherwise dominate the visual distribution.

**Two figure types** are produced:
- **Per-plate supplementary figures** (coloured by condition) — individual sample curves (α = 0.2) overlaid with per-condition group mean curves (linewidth = 2.5). These reveal whether RT instability is condition-specific or plate-wide.
- **Overview figures** (coloured by plate) — group mean curves only, with individual curves suppressed (`show_individual=False`). Generated separately for single-cell (K562, HEK293T) and QC (HeLa, site-specific QC) conditions to allow direct cross-plate comparison without per-sample noise.

#### Completeness outlier threshold

Samples are flagged as `completeness_outlier = True` if the samples don't pass the set thresholds for their respective set (e.g. SC or QC samples). The threshold of **25%** is selected from the completeness curves above — it sits below the main distribution of well-measured samples and above the cluster of clearly failed injections.

The outlier flag is propagated from the precursor to the protein AnnData by index alignment, ensuring consistent sample exclusion at both quantification levels.

In [16]:
adata_precursor.obs['completeness_outlier'] = ~(
    (adata_precursor.obs['completeness_sample_SC'] > 0.25) |
    (adata_precursor.obs['completeness_sample_QC'] > 0.25)
)

adata_protein.obs['completeness_outlier'] = (
    adata_precursor.obs['completeness_outlier']
    .reindex(adata_protein.obs_names)
)

In [ ]:
rt_props = []

conditions = ['HEK293T', 'K562', 'QC - Hela', 'Site specific QC']

for lab_name in LABS:
    rt_props_lab = compute_rt_proportions(
        adata=adata_precursor[
            (adata_precursor.obs['plate_idx']==lab_name) &
            (~adata_precursor.obs['completeness_outlier']) &
            (adata_precursor.obs['source name'].isin(conditions))
        ],
        rt_layer='EG.ApexRT',
        bins=30,
        normalize_rt=True
    )
    rt_props_lab = rt_props_lab.merge(adata_precursor.obs.reset_index(), how='left')
    rt_props.append(rt_props_lab)
    
    # Plot the retention time stability
    plot_rt_stability(
        rt_props=rt_props_lab,
        color_by='source name',
        title=lab_name,
        save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'RT_Stability'),
        filename=f"RT_stability_{lab_name}.png",
        show=SHOW_FIGURES
    )

rt_props = pd.concat(rt_props, ignore_index=True)

In [ ]:
plot_rt_stability(
    rt_props=rt_props[rt_props['source name'].isin(['K562', 'HEK293T'])],
    color_by='plate_idx',
    title=f'overview_SC',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'RT_Stability'),
    filename=f"RT_stability_overview_SC.png",
    show_individual=False,
    show=SHOW_FIGURES
)
plot_rt_stability(
    rt_props=rt_props[rt_props['source name'].isin(['Site specific QC', 'QC - Hela'])],
    color_by='plate_idx',
    title=f'overview_QC',
    save_dir=os.path.join(FIGURES_OUT, 'Figure_4', 'RT_Stability'),
    filename=f"RT_stability_overview_QC.png",
    show_individual=False,
    show=SHOW_FIGURES
)

## 3. Save

Write all four AnnData objects back to `Data/AnnData/`. The objects now carry additional columns computed in this notebook — `completeness_sample_SC/QC` in `obs`, `completeness_feature_{lab}_SC/QC` in `var`, and `completeness_outlier` in `obs` — which are consumed by downstream analysis notebooks for sample filtering and variance modelling.

In [31]:
ANNDATA_OUT = os.path.join(
    ROOT_OUT,
    'AnnData'
)

adata_precursor.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_precursor.h5ad'),
    convert_strings_to_categoricals=False
)
adata_protein.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_protein.h5ad'),
    convert_strings_to_categoricals=False
)

adata_precursor_qc.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_precursor_qc.h5ad'),
    convert_strings_to_categoricals=False
)
adata_protein_qc.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_protein_qc.h5ad'),
    convert_strings_to_categoricals=False
)